# AI-Based Detection of Hidden Electrical Energy Wastage Using LSTM Autoencoder

This Colab notebook implements the complete process documented in the micro-project report.

In [ ]:
!pip -q install tensorflow pandas numpy matplotlib seaborn scikit-learn
import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Input
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)

In [ ]:
zip_path = "household_power_consumption.zip"
extract_dir = "household_power_consumption_dir"
txt_name = "household_power_consumption.txt"
urls = [
    "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip",
    "https://raw.githubusercontent.com/saurabhg9/UCI-datasets/master/individual+household+electric+power+consumption.zip",
]
success = False
for url in urls:
    try:
        print("Downloading:", url)
        urllib.request.urlretrieve(url, zip_path)
        success = True
        break
    except Exception as error:
        print("Fallback required:", error)

if success:
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(extract_dir)
txt_path = os.path.join(extract_dir, txt_name)
if not os.path.exists(txt_path):
    txt_path = txt_name if os.path.exists(txt_name) else None
if txt_path is None:
    raise FileNotFoundError("Upload household_power_consumption.zip or household_power_consumption.txt")

In [ ]:
df_raw = pd.read_csv(txt_path, sep=";", nrows=250000, low_memory=False)
print("Raw shape:", df_raw.shape)
df_raw["Datetime"] = pd.to_datetime(df_raw["Date"] + " " + df_raw["Time"], dayfirst=True, errors="coerce")
df_raw = df_raw.dropna(subset=["Datetime"])
numeric_cols = ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
for column in numeric_cols:
    df_raw[column] = pd.to_numeric(df_raw[column], errors="coerce")
df_raw[numeric_cols] = df_raw[numeric_cols].interpolate(method="linear").ffill().bfill()
label_column = next((name for name in ("Anomaly", "anomaly", "is_anomaly", "label", "Label") if name in df_raw.columns), None)
if label_column:
    labels = df_raw[label_column].astype(str).str.strip().str.lower()
    df_raw["Anomaly"] = labels.isin({"1", "true", "yes", "anomaly"}).astype(int)
df_raw.set_index("Datetime", inplace=True)
df_resampled = df_raw[numeric_cols].resample("15min").mean().interpolate(method="linear").dropna()
if label_column:
    df_resampled["Anomaly"] = df_raw["Anomaly"].resample("15min").max().fillna(0).astype(int)
print("Resampled shape:", df_resampled.shape)

In [ ]:
n_total = len(df_resampled)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
train_df = df_resampled.iloc[:n_train]
val_df = df_resampled.iloc[n_train:n_train + n_val]
test_df = df_resampled.iloc[n_train + n_val:]

scaler = MinMaxScaler()
scaler.fit(train_df[["Global_active_power"]])
train_scaled = scaler.transform(train_df[["Global_active_power"]])
val_scaled = scaler.transform(val_df[["Global_active_power"]])
test_scaled = scaler.transform(test_df[["Global_active_power"]])

def create_sequences(data, seq_length=96):
    return np.array([data[i:i + seq_length] for i in range(len(data) - seq_length + 1)])

SEQ_LENGTH = 96
X_train = create_sequences(train_scaled, SEQ_LENGTH)
X_val = create_sequences(val_scaled, SEQ_LENGTH)
X_test = create_sequences(test_scaled, SEQ_LENGTH)
print("Train/Val/Test sequences:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
model = Sequential([
    Input(shape=(SEQ_LENGTH, 1)),
    LSTM(32, activation="relu", return_sequences=False, name="Encoder_LSTM"),
    RepeatVector(SEQ_LENGTH, name="Latent_Bridge"),
    LSTM(32, activation="relu", return_sequences=True, name="Decoder_LSTM"),
    TimeDistributed(Dense(1), name="Output_Layer"),
])
model.compile(optimizer="adam", loss="mse")
model.summary()
early_stopping = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(X_train, X_train, epochs=10, batch_size=128,
                    validation_data=(X_val, X_val), callbacks=[early_stopping], verbose=1)
model.save("lstm_autoencoder_wastage_model.keras")

In [ ]:
val_predictions = model.predict(X_val, verbose=0)
test_predictions = model.predict(X_test, verbose=0)
val_mae = np.mean(np.abs(X_val - val_predictions), axis=(1, 2))
test_mae = np.mean(np.abs(X_test - test_predictions), axis=(1, 2))
threshold = np.percentile(val_mae, 99)
anomaly_indices = np.where(test_mae > threshold)[0]
print("Threshold:", threshold)
print("Anomaly sequences:", len(anomaly_indices), "of", len(test_mae))

In [ ]:
test_actual_scaled = test_scaled[SEQ_LENGTH - 1:].reshape(-1, 1)
test_reconstructed_scaled = test_predictions[:, -1, 0].reshape(-1, 1)
count = min(len(test_actual_scaled), len(test_reconstructed_scaled))
actual_kw = scaler.inverse_transform(test_actual_scaled[:count]).ravel()
reconstructed_kw = scaler.inverse_transform(test_reconstructed_scaled[:count]).ravel()
timeline_index = test_df.index[SEQ_LENGTH - 1:SEQ_LENGTH - 1 + count]
is_anomaly = np.zeros(count, dtype=bool)
is_anomaly[anomaly_indices[anomaly_indices < count]] = True
positive_residual = actual_kw - reconstructed_kw
wastage_candidates = (positive_residual > 0) & is_anomaly
excess_kwh = np.where(wastage_candidates, positive_residual * 0.25, 0.0)
total_wastage = np.sum(excess_kwh)
print("Total estimated wastage:", round(float(total_wastage), 4), "kWh")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("LSTM Autoencoder Training")
plt.legend()
plt.show()

plt.figure(figsize=(15, 4))
plt.plot(timeline_index[:350], actual_kw[:350], label="Actual", alpha=0.75)
plt.plot(timeline_index[:350], reconstructed_kw[:350], label="Reconstructed baseline", linestyle="--")
plt.legend()
plt.title("Actual vs Reconstructed Consumption")
plt.show()

plt.figure(figsize=(15, 3.5))
plt.plot(timeline_index, actual_kw, color="gray", alpha=0.45, label="Actual power")
plt.scatter(timeline_index, np.where(is_anomaly, actual_kw, np.nan), color="red", s=12, label="Detected anomaly")
plt.legend()
plt.title("Anomaly Detection Timeline")
plt.show()

In [ ]:
test_series = pd.Series(actual_kw)
rolling_mean = test_series.rolling(window=12, min_periods=1).mean()
rolling_std = test_series.rolling(window=12, min_periods=1).std().fillna(0)
statistical_anomalies = test_series > (rolling_mean + 2.5 * rolling_std)

print("Threshold sensitivity:")
for percentile in (95, 97, 99):
    candidate = np.percentile(val_mae, percentile)
    print(percentile, "threshold =", round(float(candidate), 6), "flagged =", int(np.sum(test_mae > candidate)))
print("Statistical baseline anomalies:", int(np.sum(statistical_anomalies)))

if "Anomaly" in test_df.columns:
    true_windows = (test_df["Anomaly"].rolling(SEQ_LENGTH, min_periods=SEQ_LENGTH)
                    .max().iloc[SEQ_LENGTH - 1:].to_numpy(dtype=bool)[:len(is_anomaly)])
    print("Labeled sequence-window precision:", round(precision_score(true_windows, is_anomaly, zero_division=0), 4))
    print("Labeled sequence-window recall:", round(recall_score(true_windows, is_anomaly, zero_division=0), 4))
    print("Labeled sequence-window F1:", round(f1_score(true_windows, is_anomaly, zero_division=0), 4))
else:
    print("No anomaly labels supplied; supervised accuracy metrics are unavailable.")